## 08 — Economic Neo-Colonial Score (`econ_neocol_score`)

Constructs the directional economic neo-colonial score for each sender→recipient→year dyad.

### Formula
```
complexity_asymmetry = (ECI_sender - ECI_receiver).clip(lower=0)
trade_dependency     = bilateral_trade / receiver_GDP
econ_neocol_score    = trade_dependency * complexity_asymmetry
```

### Data sources
| Input | File | Key columns |
|---|---|---|
| ECI scores | `data/raw/economic/eci-rankings-raw.csv` | `country_iso3_code`, `year`, `eci_hs92` |
| Bilateral trade | `data/processed/v3_imf_trade.csv` | `a_iso3`, `b_iso3`, `year`, `usd_value` |
| GDP + population | `data/processed/controls/controls_merged.csv` | `iso3`, `year`, `gdp_per_capita`, `population` |
| Base panel | `data/merged/panel_with_controls_1992_2024.csv` | `sender_iso3`, `recipient_iso3`, `year` |

### Documented limitations / missingness decisions
- **1992–1994:** ECI starts 1995 — rows in those years will have `NaN` for `econ_neocol_score`. Kept as NaN, not imputed. Document as limitation.
- **68 countries with no ECI coverage:** Small island states, fragile states not in Harvard Growth Lab rankings. Rows where sender or receiver lacks ECI will be `NaN`. Not imputed.
- **receiver_GDP:** Derived as `gdp_per_capita × population` (both from `controls_merged.csv`). Per-capita GDP alone is not used.

### Output
- `data/processed/econ_neocol_score.csv` — dyadic file with `econ_neocol_score` + intermediate columns
- Later merged into main panel as edge weight for economic network layer

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

---
## Step 1 — Load ECI rankings

Using `eci_hs92` (Harvard Growth Lab, HS 1992 product classification). Zero internal missingness confirmed in audit.

In [ ]:
eci_raw = pd.read_csv('../data/raw/economic/eci-rankings-raw.csv')

eci = eci_raw[['country_iso3_code', 'year', 'eci_hs92']].copy()
eci = eci.rename(columns={'country_iso3_code': 'iso3'})

print('Shape:', eci.shape)
print('Year range:', eci['year'].min(), '–', eci['year'].max())
print('Unique countries:', eci['iso3'].nunique())
print('Missingness in eci_hs92:', eci['eci_hs92'].isnull().sum())
print()
eci.head()

---
## Step 2 — Aggregate IMF trade to bilateral total per undirected pair × year

The raw file has one row per directed flow (export or import). We need total bilateral trade between two countries in a year regardless of direction — used as the numerator in `trade_dependency`.

**Method:** sort the two ISO3 codes alphabetically to create a canonical undirected pair key, then sum all `usd_value` within that key × year.

In [ ]:
trade_raw = pd.read_csv('../data/processed/v3_imf_trade.csv')

print('Raw shape:', trade_raw.shape)
print('Year range:', trade_raw['year'].min(), '–', trade_raw['year'].max())
trade_raw.head(3)

In [ ]:
# Create canonical undirected pair key (sorted alphabetically)
trade_raw['pair_key'] = trade_raw.apply(
    lambda r: tuple(sorted([r['a_iso3'], r['b_iso3']])), axis=1
)

# Sum all usd_value per undirected pair × year
bilateral = (
    trade_raw
    .groupby(['pair_key', 'year'], as_index=False)['usd_value']
    .sum()
    .rename(columns={'usd_value': 'bilateral_trade'})
)

# Unpack pair_key into iso3_a / iso3_b (alphabetical) for merging
bilateral[['iso3_a', 'iso3_b']] = pd.DataFrame(
    bilateral['pair_key'].tolist(), index=bilateral.index
)
bilateral = bilateral.drop(columns='pair_key')
bilateral['year'] = bilateral['year'].astype(int)

print('Shape after aggregation:', bilateral.shape)
print('Year range:', bilateral['year'].min(), '–', bilateral['year'].max())
print()
bilateral.head()

---
## Step 3 — Derive receiver_GDP from controls

`receiver_GDP = gdp_per_capita × population` (both from `controls_merged.csv`).

This gives total nominal GDP in USD (World Bank). Used as denominator in `trade_dependency`.

In [ ]:
controls = pd.read_csv('../data/processed/controls/controls_merged.csv')

print('Shape:', controls.shape)
print('Columns:', list(controls.columns))
controls.head(3)

In [ ]:
controls['receiver_GDP'] = controls['gdp_per_capita'] * controls['population']

gdp_lookup = controls[['iso3', 'year', 'receiver_GDP']].copy()

missing_gdp = gdp_lookup['receiver_GDP'].isnull().sum()
total = len(gdp_lookup)
print(f'receiver_GDP missingness: {missing_gdp} / {total} ({missing_gdp/total*100:.1f}%)')
print('  → driven by gdp_per_capita (406 missing) and population (66 missing) in controls')
print()
gdp_lookup.head()

---
## Step 4 — Load base panel

Using `panel_with_controls_1992_2024.csv` as the dyadic skeleton. We merge everything onto this.

In [ ]:
panel = pd.read_csv('../data/merged/panel_with_controls_1992_2024.csv')

print('Shape:', panel.shape)
print('Columns:', list(panel.columns))
print('Year range:', panel['year'].min(), '–', panel['year'].max())
print()
panel.head(3)

---
## Step 5 — Merge sender ECI onto panel

In [ ]:
eci_sender = eci.rename(columns={'iso3': 'sender_iso3', 'eci_hs92': 'eci_sender'})

panel = panel.merge(eci_sender, on=['sender_iso3', 'year'], how='left')

print('Shape after sender ECI merge:', panel.shape)
print(f'eci_sender missingness: {panel["eci_sender"].isnull().sum()} / {len(panel)} ({panel["eci_sender"].isnull().mean()*100:.1f}%)')
print()
panel[['sender_iso3', 'recipient_iso3', 'year', 'eci_sender']].head()

---
## Step 6 — Merge receiver ECI onto panel

In [ ]:
eci_receiver = eci.rename(columns={'iso3': 'recipient_iso3', 'eci_hs92': 'eci_receiver'})

panel = panel.merge(eci_receiver, on=['recipient_iso3', 'year'], how='left')

print('Shape after receiver ECI merge:', panel.shape)
print(f'eci_receiver missingness: {panel["eci_receiver"].isnull().sum()} / {len(panel)} ({panel["eci_receiver"].isnull().mean()*100:.1f}%)')
print()
panel[['sender_iso3', 'recipient_iso3', 'year', 'eci_sender', 'eci_receiver']].head()

---
## Step 7 — Merge bilateral trade onto panel

The bilateral trade table is undirected (iso3_a < iso3_b alphabetically). We create the same canonical pair key from the panel to join on.

In [ ]:
# Build same canonical pair key on the panel
panel['iso3_a'] = panel.apply(lambda r: min(r['sender_iso3'], r['recipient_iso3']), axis=1)
panel['iso3_b'] = panel.apply(lambda r: max(r['sender_iso3'], r['recipient_iso3']), axis=1)

panel = panel.merge(
    bilateral[['iso3_a', 'iso3_b', 'year', 'bilateral_trade']],
    on=['iso3_a', 'iso3_b', 'year'],
    how='left'
)

panel = panel.drop(columns=['iso3_a', 'iso3_b'])

print('Shape after bilateral trade merge:', panel.shape)
print(f'bilateral_trade missingness: {panel["bilateral_trade"].isnull().sum()} / {len(panel)} ({panel["bilateral_trade"].isnull().mean()*100:.1f}%)')
print('  → NaN means no recorded trade flow between this pair in this year')
print()
panel[['sender_iso3', 'recipient_iso3', 'year', 'bilateral_trade']].head()

---
## Step 8 — Merge receiver_GDP onto panel

In [ ]:
gdp_lookup_recv = gdp_lookup.rename(columns={'iso3': 'recipient_iso3'})

panel = panel.merge(gdp_lookup_recv, on=['recipient_iso3', 'year'], how='left')

print('Shape after receiver_GDP merge:', panel.shape)
print(f'receiver_GDP missingness: {panel["receiver_GDP"].isnull().sum()} / {len(panel)} ({panel["receiver_GDP"].isnull().mean()*100:.1f}%)')
print()
panel[['sender_iso3', 'recipient_iso3', 'year', 'bilateral_trade', 'receiver_GDP']].head()

---
## Step 9 — Compute `econ_neocol_score`

```
complexity_asymmetry = (ECI_sender - ECI_receiver).clip(lower=0)
trade_dependency     = bilateral_trade / receiver_GDP
econ_neocol_score    = trade_dependency * complexity_asymmetry
```

NaN propagation is intentional:
- Missing ECI (either side) → NaN
- Missing bilateral_trade → NaN (no trade relationship recorded)
- Missing receiver_GDP → NaN

In [ ]:
panel['complexity_asymmetry'] = (panel['eci_sender'] - panel['eci_receiver']).clip(lower=0)
panel['trade_dependency']     = panel['bilateral_trade'] / panel['receiver_GDP']
panel['econ_neocol_score']    = panel['trade_dependency'] * panel['complexity_asymmetry']

print('Shape:', panel.shape)
print()
print('=== Missingness summary for computed columns ===')
for col in ['complexity_asymmetry', 'trade_dependency', 'econ_neocol_score']:
    n = panel[col].isnull().sum()
    print(f'  {col}: {n} NaN ({n/len(panel)*100:.1f}%)')

print()
print('=== Non-null econ_neocol_score distribution ===')
print(panel['econ_neocol_score'].describe())

print()
panel[['sender_iso3', 'recipient_iso3', 'year',
       'eci_sender', 'eci_receiver',
       'complexity_asymmetry', 'bilateral_trade',
       'receiver_GDP', 'trade_dependency',
       'econ_neocol_score']].head(10)

---
## Step 10 — Sanity checks

In [ ]:
# Check 1: complexity_asymmetry should never be negative
neg = (panel['complexity_asymmetry'] < 0).sum()
print(f'Negative complexity_asymmetry values: {neg}  (should be 0)')

# Check 2: North-North dyads (both high ECI) should produce asymmetry ~0
# Example: USA → GBR
nn = panel[(panel['sender_iso3'] == 'USA') & (panel['recipient_iso3'] == 'GBR')]
print()
print('USA → GBR (should have low/zero complexity_asymmetry):')
print(nn[['year', 'eci_sender', 'eci_receiver', 'complexity_asymmetry', 'econ_neocol_score']].tail(5))

# Check 3: High-asymmetry example — USA → low-complexity country
# Find a recipient with consistently low ECI
low_eci = panel[panel['eci_receiver'] < -1.5]
example_receiver = low_eci['recipient_iso3'].value_counts().index[0]
usa_low = panel[(panel['sender_iso3'] == 'USA') & (panel['recipient_iso3'] == example_receiver)]
print()
print(f'USA → {example_receiver} (should have high complexity_asymmetry):')
print(usa_low[['year', 'eci_sender', 'eci_receiver', 'complexity_asymmetry', 'econ_neocol_score']].tail(5))

In [ ]:
# Check 4: Missingness breakdown by cause
null_score = panel['econ_neocol_score'].isnull()

null_eci_only   = null_score & (panel['eci_sender'].isnull() | panel['eci_receiver'].isnull()) & panel['bilateral_trade'].notnull()
null_trade_only = null_score & panel['eci_sender'].notnull() & panel['eci_receiver'].notnull() & panel['bilateral_trade'].isnull()
null_gdp_only   = null_score & panel['bilateral_trade'].notnull() & panel['receiver_GDP'].isnull()
null_pre1995    = null_score & (panel['year'] < 1995)

print('Missingness breakdown for econ_neocol_score NaNs:')
print(f'  Total NaN rows:              {null_score.sum()}')
print(f'  Due to year < 1995 (ECI gap): {null_pre1995.sum()}')
print(f'  Due to missing ECI only:      {null_eci_only.sum()}')
print(f'  Due to missing trade only:    {null_trade_only.sum()}')
print(f'  Due to missing GDP only:      {null_gdp_only.sum()}')

---
## Step 11 — Save output

Saving two files:
1. **`econ_neocol_score.csv`** — lean file with just the score and its components, for merging downstream
2. The full panel with score attached is **not** saved here — merging into the main panel is done in the next notebook

In [ ]:
output_cols = [
    'sender_iso3', 'recipient_iso3', 'year',
    'eci_sender', 'eci_receiver',
    'complexity_asymmetry',
    'bilateral_trade',
    'receiver_GDP',
    'trade_dependency',
    'econ_neocol_score'
]

output = panel[output_cols].copy()

output.to_csv('../data/processed/econ_neocol_score.csv', index=False)

print('Saved: data/processed/econ_neocol_score.csv')
print('Shape:', output.shape)
print()
print('Non-null econ_neocol_score rows:', output['econ_neocol_score'].notnull().sum())
print('NaN rows:', output['econ_neocol_score'].isnull().sum())
print()
output.head(10)